# MedNorm-VI — VietMed-NER Parquet Preprocessing (Colab CPU; real, not a placeholder)

**Not model training. CPU is sufficient (no GPU).** Converts VietMed-NER Parquet into
canonical half-open JSONL using the tracked adapter (`mednorm_vi.data_engine.vietmed_ner`),
resolves the word/BIO-tag columns by name AND type, applies the versioned mapping,
**excludes audio**, and writes hashed artifacts. Fails fast on any integrity problem.


## 1. Runtime detection · 2. GPU/RAM/disk report


In [ ]:
import sys
import platform
import shutil

print('python', platform.python_version(), 'in_colab', 'google.colab' in sys.modules)
print('disk_free_gb', round(shutil.disk_usage('/')[2] / 1e9, 1))


## 3. Paths — separate Drive (persistent) from repo (Colab temp)
`REPO_DIR` is a Colab-temporary git clone; raw data and returned artifacts persist on Drive.


In [ ]:
DRIVE_ROOT = '/content/drive/MyDrive/MedNorm-VI'          # <-- EDIT (Drive persistent root)
REPO_DIR = '/content/MedNorm-VI'                          # git clone in Colab temp storage
VIETMED_SOURCE_DIR = DRIVE_ROOT + '/data/external/public_ner/vietmed_ner'
VIETMED_PARQUET_DIR = VIETMED_SOURCE_DIR + '/data'
ARTIFACT_DIR = DRIVE_ROOT + '/data/derived/training_corpora/vietmed_ner_v1'
try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError as exc:
    print('not in Colab:', exc)


## 4. Pinned pyarrow installation (Colab)


In [ ]:
PYARROW_PIN = 'pyarrow==17.0.0'
# !pip -q install {PYARROW_PIN}
import pyarrow  # noqa: E402
print('pyarrow', pyarrow.__version__)


## 5. Repository clone + adapter import (Colab temp REPO_DIR)


In [ ]:
import os
import sys

# !git clone <repo-url> {REPO_DIR}   # clone into Colab temp storage
if REPO_DIR + '/src' not in sys.path:
    sys.path.insert(0, REPO_DIR + '/src')
from mednorm_vi.data_engine import vietmed_ner as vm
print('adapter', vm.ADAPTER_VERSION)


## 6. Commit / mapping verification


In [ ]:
import subprocess

EXPECTED_COMMIT = '<fill: reviewed repo HEAD>'
try:
    head = subprocess.check_output(['git', '-C', REPO_DIR, 'rev-parse', 'HEAD']).decode().strip()
    print('repo HEAD', head, 'expected', EXPECTED_COMMIT)
except Exception as exc:
    head = EXPECTED_COMMIT
    print('git check skipped:', exc)
mapping = vm.load_vietmed_mapping(REPO_DIR + '/' + vm.DEFAULT_MAPPING)
print('mapping v', mapping.version, 'concrete', mapping.type_mapping)


## 7. Parquet discovery · 8. source hashing


In [ ]:
import hashlib

assert os.path.isdir(VIETMED_PARQUET_DIR), 'missing: ' + VIETMED_PARQUET_DIR
parquets = sorted(f for f in os.listdir(VIETMED_PARQUET_DIR) if f.endswith('.parquet'))
assert parquets, 'no .parquet files found (fail fast)'
def sha256(path):
    h = hashlib.sha256()
    with open(path, 'rb') as fh:
        for chunk in iter(lambda: fh.read(1 << 20), b''):
            h.update(chunk)
    return h.hexdigest()
source_hashes = {f: sha256(VIETMED_PARQUET_DIR + '/' + f) for f in parquets}
print('parquet files', parquets)


## 9. Column resolution (words/tokens ; labels/ner_tags/string-tags) — fail on ambiguity


In [ ]:
import pyarrow.parquet as pq  # noqa: E402

schema = pq.read_schema(VIETMED_PARQUET_DIR + '/' + parquets[0])
column_types = {f.name: str(f.type) for f in schema}
word_col, tag_col = vm.resolve_bio_columns(column_types)   # raises if missing/ambiguous
resolved_columns = {'word_col': word_col, 'tag_col': tag_col}
print('resolved columns', resolved_columns, '| all columns', sorted(column_types))


## 10. Read (audio NOT read) + convert to canonical half-open JSONL


In [ ]:
rows = vm.read_vietmed_parquet(VIETMED_PARQUET_DIR)   # resolves + reads word/tag cols only
examples, summary = vm.convert_rows(rows, mapping=mapping)
print('rows', summary['rows_in'], 'examples', summary['examples_emitted'],
      'entities', summary['entities_emitted'])
print('accepted', summary['accepted'], 'repaired', summary['deterministic_repair'],
      'excluded', summary['excluded'], 'human_review', summary['human_review_required'])


## 11. Fail-fast: offsets valid AND no HUMAN_REVIEW_REQUIRED · 12. determinism


In [ ]:
assert summary['offset_invalid'] == 0, 'offset invariant violated (fail fast)'
assert summary['human_review_required'] == 0, 'words/tags mismatch -> human review (fail fast)'
examples2, _ = vm.convert_rows(rows, mapping=mapping)
assert examples == examples2, 'non-deterministic conversion (fail fast)'
print('offset_invalid 0; human_review 0; deterministic', examples == examples2)


## 13. Quality reports + 14. artifact writing (clear names; resolved columns recorded)


In [ ]:
manifest = vm.write_artifacts(ARTIFACT_DIR, examples, summary, mapping=mapping,
                              source_hashes=source_hashes, repo_commit=head,
                              resolved_columns=resolved_columns)
print('resolved_columns in manifest', manifest['resolved_columns'])
print('examples_jsonl_sha256', manifest['examples_jsonl_sha256'])


## 15. SHA-256 verification (re-load + hash check)


In [ ]:
reloaded = vm.load_vietmed_artifacts(ARTIFACT_DIR)   # raises if hash mismatch
assert reloaded == examples, 'artifact roundtrip mismatch (fail fast)'
print('artifact hash verified; rows', len(reloaded))


## 16. Final artifact tree (real files, sizes, hashes)


In [ ]:
print('audio_excluded', manifest['audio_excluded'])
for root, _dirs, files in os.walk(ARTIFACT_DIR):
    for f in sorted(files):
        p = os.path.join(root, f)
        print(round(os.path.getsize(p) / 1024, 1), 'KB', sha256(p)[:16], p)


## 17. Return to repository
Copy `ARTIFACT_DIR` (on Drive) into the repo at
`data/derived/training_corpora/vietmed_ner_v1/` (git-ignored). Then locally rebuild the
governed corpus INCLUDING VietMed:

```bash
env PYTHONPATH=src python3 -m mednorm_vi.data_engine.cli build-governed-corpus
```

The builder auto-detects the artifacts, validates the manifest hash, includes VietMed
(MAP_APPROXIMATE → train only), rebuilds families/splits, and reports
`vietmed_status: included_from_artifacts`. No raw Parquet/pyarrow needed afterward; no audio.
